# P4 — Ngày 5: Decision Tree (Baseline B)

Chủ sở hữu: **P4 Model Engineer**

Đường chạy: `parquet → DT → xác suất → ngưỡng → NMS → mAP` (KE_HOACH.md §2).

> ⚠️ Chỉ số của dự án là **mAP_macro** ở bước [4/4], KHÔNG phải accuracy cửa sổ.
> Sàn phải vượt trên val: **Baseline A = 0.5176** (KE_HOACH.md §4).

> ⚠️ **Không chạy trên test** — test chỉ mở Ngày 12, đúng một lần (§8 quy tắc 3).

## 1. Môi trường

`BRANCH` phải là branch chứa `train_model.py` + `detect.py`. Đổi về `main` sau khi đã merge.

In [ ]:
BRANCH = 'feature/plan'   # <<< đổi thành 'main' sau khi merge
REPO   = 'https://github.com/AIVIETNAM-AIO-thanhnhan/car-parkinglot-count.git'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip -q install pyarrow
    # clone lần đầu; các lần sau chỉ fetch — luôn ép về đúng branch để không chạy nhầm code cũ
    ![ -d /content/code ] || git clone -q $REPO /content/code
    !cd /content/code && git fetch -q origin $BRANCH && git checkout -q $BRANCH && git reset -q --hard origin/$BRANCH && git log --oneline -1
    SRC = '/content/code/src'
except ImportError:
    from pathlib import Path
    SRC = str(Path.cwd().parent / 'src')

import sys
if SRC not in sys.path: sys.path.insert(0, SRC)
import config
print('sẵn sàng —', 'Colab' if config.ON_COLAB else 'local')
print('DRIVE =', config.DRIVE)

## 2. Kiểm tra dữ liệu trên Drive

Cần đúng 2 thứ, cả hai nằm dưới `MyDrive/pklot_project/`:

| Đường dẫn | Nội dung | Ai tạo |
|---|---|---|
| `features/shard_*.parquet` | 8 shard, 1,8 GB | P2/P3 (Ngày 3) |
| `processed/gt.csv` | 34.686 dòng nhãn | sinh lại được — ô kế tiếp |

`gt.csv` **không nằm trong Git** (dữ liệu dẫn xuất, report §6). Nếu thiếu, ô kế tiếp sinh lại —
nhưng bước đó cần **ảnh gốc PKLot** trong `raw/`. Không có raw thì nhờ P2 copy thẳng `gt.csv`
(~4 MB) vào `processed/`, nhanh hơn nhiều so với tải lại 2 GB ảnh.

In [ ]:
from pathlib import Path
import pandas as pd

shards = sorted(config.FEAT.glob('shard_*.parquet'))
print(f'{len(shards)} shard trong {config.FEAT}')
for p in shards:
    print(f'  {p.name}  {p.stat().st_size/1e6:,.0f} MB')
assert shards, f'Chưa thấy shard nào. Kiểm tra {config.FEAT} trên Drive.'

gt_path = config.PROC / 'gt.csv'
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    print(f'\ngt.csv: {len(gt):,} dòng')
    print(gt.groupby('split').image_id.nunique().to_string(), '  <-- phải là 340/29/210 ảnh')
else:
    print('\n⚠️ Thiếu processed/gt.csv — chạy ô kế tiếp để sinh lại (cần raw/PKLot).')

In [ ]:
# CHỈ chạy khi thiếu gt.csv VÀ đã có ảnh gốc trong raw/. Mất vài chục giây (chỉ parse XML).
if not gt_path.exists():
    import pklot_data
    assert pklot_data.LOT_ROOT.exists(), (
        f'Không có ảnh gốc ở {pklot_data.LOT_ROOT}. '
        'Nhờ P2 copy processed/gt.csv vào Drive thay vì tải lại 2 GB.')
    rows = pklot_data.build_gt_rows(pklot_data.load_split())
    gt_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(gt_path, index=False)
    print(f'đã sinh {gt_path}: {len(rows):,} dòng')

## 3. Cổng kiểm tra — chạy TRƯỚC khi train

Ba self-test phải PASS hết. FAIL nghĩa là mọi số đo sau đó không đáng tin (KE_HOACH.md §8 quy tắc 1).

In [ ]:
import evaluate_pklot, baselines, detect

evaluate_pklot.self_test()   # harness chấm điểm
detect.self_test()           # NMS + lọc lớp nền
baselines.self_test()        # chống tái phát bug xê dịch camera

## 4. Train Decision Tree

Chạy qua CLI để tham số được ghi đúng vào `results.csv`.

**RAM Colab free ~12,7 GB.** 1,18 triệu cửa sổ × 395 feature ≈ 1,9 GB, cộng bản sao float32 lúc
train là ~4 GB — vẫn vừa, nhưng **lần chạy đầu nên dùng `--sample 0.25`** để biết đường chạy thông
suốt trong vài phút rồi mới chạy full. `--sample` chỉ lấy mẫu TRAIN (phân tầng theo ảnh + lớp);
val luôn giữ 100% dòng nên các lần chạy vẫn so sánh được với nhau.

In [ ]:
!cd {SRC} && python train_model.py --sample 0.25 --print-rules 3 --no-log

Chạy thật trên toàn bộ train — dòng này mới được ghi vào `results.csv`:

In [ ]:
!cd {SRC} && python train_model.py --print-rules 3 --sweep-threshold

## 5. Đọc kết quả

| Nhìn ở đâu | Ý nghĩa |
|---|---|
| `[3/4]` accuracy cửa sổ | **chỉ tham khảo**. So với dòng "đoán bừa lớp đông nhất" ngay dưới nó — hơn ít nghĩa là chưa học được gì |
| `[4/4]` `mAP_macro` | **chỉ số dự án**, điền vào bảng KE_HOACH §5 |
| dòng so sàn | phải vượt **0.5176** thì DT mới hơn việc chỉ nhớ vị trí |
| tổng importance theo nhóm | §9 dự đoán `color_*` quan trọng hơn `hog_*` — kiểm chứng ở đây |

Ngày 5 mAP còn thấp là **đúng kỳ vọng** (§7). Random Forest + hard negative mining mới là chỗ điểm lên.

In [ ]:
res = pd.read_csv(Path(SRC).parent / 'results.csv')
res[res.experiment.str.contains('Decision Tree')][
    ['date', 'split', 'mAP_macro', 'AP_occupied', 'AP_empty', 'free_slots_MAE', 'train_time']]

## 6. Nhìn ảnh — 30 phút, cả nhóm (KE_HOACH.md §7 Ngày 5)

Vẽ box lên 5 ảnh val, xem false positive rơi vào đâu (lối đi? vỉa hè? bóng cây?).
Buổi này định hướng cả tuần sau — đừng bỏ.

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import pklot_data, train_model

df = train_model.load_features(splits=('train', 'val'))
tr, ev = df[df.split == 'train'], df[df.split == 'val']
Xtr, ytr, cols = train_model.split_xy(tr)
clf, _ = train_model.train_decision_tree(Xtr, ytr)
gt = train_model.load_gt('val')
_, pred, _, _ = train_model.evaluate_detection(clf, ev, gt)

crops = json.load(open(config.PROC / 'crops.json'))
paths = {Path(p).stem: p for p in pklot_data.load_split()['val']}
COLOR = {0: 'lime', 1: 'red'}   # 0 = ô trống, 1 = có xe

for image_id in sorted(pred.image_id.unique())[:5]:
    lot = pklot_data._lot_of(paths[image_id])
    x0, y0, x1, y1 = crops[lot]
    img = Image.open(paths[image_id]).convert('RGB').crop((x0, y0, x1, y1))
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    panels = [('THẬT', gt[gt.image_id == image_id]), ('ĐOÁN', pred[pred.image_id == image_id])]
    for ax, (title, rows) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(f'{image_id} — {title} ({len(rows)} box)')
        ax.axis('off')
        for r in rows.itertuples():
            if r.label not in COLOR:
                continue   # bỏ ô không rõ nhãn (-1)
            ax.add_patch(plt.Rectangle((r.x_min, r.y_min), r.x_max - r.x_min, r.y_max - r.y_min,
                                       fill=False, color=COLOR[r.label], lw=1.5))
    plt.tight_layout()
    plt.show()